In [ ]:
!python3 -V
import pDESy
from IPython.display import Markdown
from pDESy.model.base_project import BaseProject
from pDESy.model.base_project import datetime
from pDESy.model.base_product import BaseProduct
from pDESy.model.base_component import BaseComponent
from pDESy.model.base_workflow import BaseWorkflow
from pDESy.model.base_task import BaseTask
from pDESy.model.base_task import BaseTaskDependency
from pDESy.model.base_team import BaseTeam
from pDESy.model.base_worker import BaseWorker
from pDESy.model.base_facility import BaseFacility
from pDESy.model.base_workplace import BaseWorkplace
from pDESy.model.base_priority_rule import TaskPriorityRuleMode,ResourcePriorityRuleMode

Python 3.13.5


In [117]:
pDESy.__version__

'0.7.3'

製品定義

In [118]:
project = BaseProject("sample_workflow")
project = BaseProject(init_datetime = datetime.datetime(2025, 1, 1, 0, 0, 0), unit_timedelta=datetime.timedelta(minutes=240))

product = project.create_product("product")

A = product.create_component("A")
B = product.create_component("B")

ワークフロー定義・同一ワークフロー間依存関係

In [119]:
workflowA = project.create_workflow("workflowA")

task_A1 = workflowA.create_task("A1", default_work_amount=4.0)
task_A2 = workflowA.create_task("A2", default_work_amount=4.0)
task_A3 = workflowA.create_task("A3", default_work_amount=4.0)

A.update_targeted_task_set({task_A1, task_A2, task_A3})

task_A2.add_input_task(task_A1)
task_A3.add_input_task(task_A2)

In [120]:
workflowB = project.create_workflow("workflowB")

task_B1 = workflowB.create_task("B1", default_work_amount=4.0)
task_B2 = workflowB.create_task("B2", default_work_amount=4.0)
task_B3 = workflowB.create_task("B3", default_work_amount=4.0)

B.update_targeted_task_set({task_B1, task_B2, task_B3})

task_B2.add_input_task(task_B1)
task_B3.add_input_task(task_B2)

異なるワークフロー間依存関係

In [121]:
task_B1.add_input_task(task_A2)

設備・人員

In [122]:
# wrokplace model
placeA = project.create_workplace("placeA", max_space_size=10.0)
placeB = project.create_workplace("placeB", max_space_size=10.0)

facilityA = placeA.create_facility("facilityA", cost_per_time=1)
facilityB = placeB.create_facility("facilityB", cost_per_time=1)
facilityA.workamount_skill_mean_map = {task_A1.name:1.0,task_A2.name:1.0, task_A3.name:1.0} 
facilityB.workamount_skill_mean_map = {task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0} 

#team model
team = project.create_team("team1")

wA = team.create_worker("workerA", cost_per_time=10.0)
wB = team.create_worker("workerB", cost_per_time=10.0)

wA.workamount_skill_mean_map = {
    task_A1.name:1.0,task_A2.name:1.0, task_A3.name:1.0,
    task_B1.name:1.0,task_B2.name:1.0,
    } 
wB.workamount_skill_mean_map = {
    task_A1.name:1.0,task_A2.name:1.0,
    task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0
    }

wA.facility_skill_map = {facilityA.name:1.0, facilityB.name:1.0}
wB.facility_skill_map = wA.facility_skill_map.copy()

team.update_targeted_task_set({task_A1,task_A2,task_A3, task_B1,task_B2,task_B3})

placeA.update_targeted_task_set({task_A1,task_A2,task_A3})
placeB.update_targeted_task_set({task_B1,task_B2,task_B3})

In [123]:
project.simulate(max_time=200, progress_bar=True)

Completed:   7%|▋         | 14/200 [00:00<00:00, 10688.07time/s] 


In [124]:
workflowA.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [125]:
workflowB.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [126]:
from plotly.figure_factory import create_gantt

task_id_list = [
    task_A1.ID, task_A2.ID, task_A3.ID,
    task_B1.ID, task_B2.ID, task_B3.ID
]

all_workflows = [
    workflowA,
    workflowB
]

combined_df = []
for wf in all_workflows:
    combined_df.extend(
        wf.create_data_for_gantt_plotly(
            init_datetime=project.init_datetime,
            unit_timedelta=project.unit_timedelta,
            target_id_order_list=list(task_id_list),
            print_workflow_name=True,
            view_ready=False,           # READY も表示したいなら True
            finish_margin=1.0
        )
    )

colors = {"WORKING": "rgb(146, 237, 5)", "READY": "rgb(107,127,135)"}

fig = create_gantt(
    combined_df,
    title="All Workflows Gantt",
    colors=colors,
    index_col="State",
    showgrid_x=True,
    showgrid_y=True,
    group_tasks=True,
    show_colorbar=True,
)

fig.show()

In [127]:
team.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [128]:
diagram = "flowchart TB\n" + "\n".join(project.get_mermaid_diagram())
display(Markdown(f"```mermaid\n{diagram}\n```"))

```mermaid
flowchart TB
subgraph c01fefc8-9347-4621-8aed-ec9bffbe76ee[product]
direction LR
d04f9086-899d-4530-a55e-c176b1146a0e@{shape: odd, label: 'A'}
d7183a26-696f-4fba-993b-0691a9da31e6@{shape: odd, label: 'B'}
end
subgraph cd5d7fb7-cbaf-49a0-8131-720c0d5ee99b[workflowA]
direction LR
89a423c9-197f-4f80-b505-03ba7cedd167@{shape: rect, label: 'A3<br>0.0'}
57c99beb-7340-4ac5-ab4b-9443f7a6011c@{shape: rect, label: 'A1<br>0.0'}
a37da956-51d3-4a3c-ab5f-edddc40268dd@{shape: rect, label: 'A2<br>0.0'}
a37da956-51d3-4a3c-ab5f-edddc40268dd-->89a423c9-197f-4f80-b505-03ba7cedd167
57c99beb-7340-4ac5-ab4b-9443f7a6011c-->a37da956-51d3-4a3c-ab5f-edddc40268dd
end
subgraph 8d4d88ba-b72c-4fb8-a2c1-177d403c97d8[workflowB]
direction LR
25e1d822-90c2-4e12-bc8e-e2d5d089d31c@{shape: rect, label: 'B2<br>0.0'}
a1a7151b-af47-46f2-976d-abaadb4f08bb@{shape: rect, label: 'B1<br>0.0'}
7ceb104e-6951-457d-bb9f-4f1b657899a7@{shape: rect, label: 'B3<br>0.0'}
a1a7151b-af47-46f2-976d-abaadb4f08bb-->25e1d822-90c2-4e12-bc8e-e2d5d089d31c
25e1d822-90c2-4e12-bc8e-e2d5d089d31c-->7ceb104e-6951-457d-bb9f-4f1b657899a7
end
subgraph f80c1807-3d9f-4569-958a-dacad74a640b[team1]
direction LR
1609c142-220a-4b7b-95c3-9c625d74e30b@{shape: stadium, label: 'workerA'}
f7e64aa7-e791-4235-ae08-4769da335869@{shape: stadium, label: 'workerB'}
end
subgraph f4a9b4ca-e8ff-4aca-9dd7-fc3648994f50[placeA]
direction LR
418f39d6-5f2c-4827-a6f8-5a26cd8ea393@{shape: stadium, label: 'facilityA'}
end
subgraph 50ed0969-a2b1-4b22-b62d-ac3fb0e5e0a2[placeB]
direction LR
98848852-eced-402f-93de-23a412d4a22e@{shape: stadium, label: 'facilityB'}
end
d04f9086-899d-4530-a55e-c176b1146a0e-.-89a423c9-197f-4f80-b505-03ba7cedd167
d04f9086-899d-4530-a55e-c176b1146a0e-.-57c99beb-7340-4ac5-ab4b-9443f7a6011c
d04f9086-899d-4530-a55e-c176b1146a0e-.-a37da956-51d3-4a3c-ab5f-edddc40268dd
d7183a26-696f-4fba-993b-0691a9da31e6-.-a1a7151b-af47-46f2-976d-abaadb4f08bb
d7183a26-696f-4fba-993b-0691a9da31e6-.-25e1d822-90c2-4e12-bc8e-e2d5d089d31c
d7183a26-696f-4fba-993b-0691a9da31e6-.-7ceb104e-6951-457d-bb9f-4f1b657899a7
89a423c9-197f-4f80-b505-03ba7cedd167-.-f80c1807-3d9f-4569-958a-dacad74a640b
f4a9b4ca-e8ff-4aca-9dd7-fc3648994f50-.-89a423c9-197f-4f80-b505-03ba7cedd167
57c99beb-7340-4ac5-ab4b-9443f7a6011c-.-f80c1807-3d9f-4569-958a-dacad74a640b
f4a9b4ca-e8ff-4aca-9dd7-fc3648994f50-.-57c99beb-7340-4ac5-ab4b-9443f7a6011c
a37da956-51d3-4a3c-ab5f-edddc40268dd-.-f80c1807-3d9f-4569-958a-dacad74a640b
f4a9b4ca-e8ff-4aca-9dd7-fc3648994f50-.-a37da956-51d3-4a3c-ab5f-edddc40268dd
25e1d822-90c2-4e12-bc8e-e2d5d089d31c-.-f80c1807-3d9f-4569-958a-dacad74a640b
50ed0969-a2b1-4b22-b62d-ac3fb0e5e0a2-.-25e1d822-90c2-4e12-bc8e-e2d5d089d31c
a1a7151b-af47-46f2-976d-abaadb4f08bb-.-f80c1807-3d9f-4569-958a-dacad74a640b
50ed0969-a2b1-4b22-b62d-ac3fb0e5e0a2-.-a1a7151b-af47-46f2-976d-abaadb4f08bb
7ceb104e-6951-457d-bb9f-4f1b657899a7-.-f80c1807-3d9f-4569-958a-dacad74a640b
50ed0969-a2b1-4b22-b62d-ac3fb0e5e0a2-.-7ceb104e-6951-457d-bb9f-4f1b657899a7
a37da956-51d3-4a3c-ab5f-edddc40268dd-->a1a7151b-af47-46f2-976d-abaadb4f08bb
```